In [1]:
import os
import collections
import numpy as np
import tensorflow as tf

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ===== 1. Caricamento EMNIST da tensorflow_datasets (senza TFF) =====
import tensorflow_datasets as tfds

NUM_CLIENTS = 5
NUM_ROUNDS = 10
EPOCHS = 10
BATCH_SIZE = 20
SHUFFLE_BUFFER = 100
PREFETCH_BUFFER = 10

print("Caricamento EMNIST...")
emnist_data = tfds.load("emnist/byclass", split="train", as_supervised=True)

# ===== 2. Preprocessamento e creazione client dataset =====
def preprocess(images, labels):
    images = tf.reshape(images, [-1, 784])
    images = tf.cast(images, tf.float32) / 255.0
    labels = tf.cast(labels, tf.int32)
    return images, labels

def prepare_client_datasets(dataset, num_clients, samples_per_client=100):
    dataset = dataset.shuffle(10000)
    dataset = dataset.filter(lambda x, y: tf.less(y, 10))  # ⬅️ filtra etichette 0–9
    dataset = dataset.batch(1)
    all_data = list(tfds.as_numpy(dataset))[:samples_per_client * num_clients]

    clients = []
    for i in range(num_clients):
        client_samples = all_data[i*samples_per_client:(i+1)*samples_per_client]
        x_client = np.array([tf.cast(tf.reshape(x, [-1]), tf.float32) / 255.0 for x, _ in client_samples])
        y_client = np.array([y for _, y in client_samples])
        clients.append((x_client, y_client))

    return clients

clients_data = prepare_client_datasets(emnist_data, NUM_CLIENTS)

# ===== 3. Definizione modello base =====
def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(784,)),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='sgd',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def get_model_weights(model):
    return model.get_weights()

def set_model_weights(model, weights):
    model.set_weights(weights)

def average_weights(weights_list):
    avg_weights = []
    for weights in zip(*weights_list):
        avg_weights.append(np.mean(weights, axis=0))
    return avg_weights

# ===== 4. Ciclo federato manuale =====
global_model = create_model()

for round_num in range(1, NUM_ROUNDS + 1):
    print(f"\n🔁 Federated Round {round_num}")

    client_weights = []

    for client_idx, (x, y) in enumerate(clients_data):
        client_model = create_model()
        set_model_weights(client_model, get_model_weights(global_model))

        client_model.fit(x, y,
                         batch_size=BATCH_SIZE,
                         epochs=EPOCHS,
                         verbose=0)

        client_weights.append(get_model_weights(client_model))

    new_global_weights = average_weights(client_weights)
    set_model_weights(global_model, new_global_weights)

    # Valutazione su tutti i dati (centrale, per monitoraggio)
    all_x = np.concatenate([x for x, _ in clients_data])
    all_y = np.concatenate([y for _, y in clients_data])
    loss, acc = global_model.evaluate(all_x, all_y, verbose=1)
    print(f"✅ Global Accuracy after round {round_num}: {acc:.4f}")


2025-07-21 13:35:16.794966: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-21 13:35:16.795042: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-21 13:35:16.939152: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-21 13:35:17.210780: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-21 13:35:18.659894: W tensorflow/comp

ModuleNotFoundError: No module named 'tensorflow_datasets'